# CSD Lakehouse — Keşif Not Defteri

Bu defter, Nessie kataloğuna bağlı hazır bir SparkSession ile gelir.
Veri bilimci / analist keşfi için tasarlandı — ETL için `jobs/` altındaki
scriptleri kullanın.

**Başlatma:** `docker compose --profile notebook up -d` → http://localhost:8888

In [ ]:
import sys
sys.path.insert(0, "/opt/spark/jobs")
from common.session import build_spark, CATALOG

# Katalog ayarlarının tamamı jobs/common/session.py içinde.
# Tek kaynak: notebook ile ETL job'ları AYNI konfigürasyonu kullanır,
# dolayısıyla burada gördüğünüz sonuç üretimde de aynıdır.
spark = build_spark("jupyter_kesif", branch="main")
spark

## Katalogda ne var?

In [ ]:
for ns in ["bronze", "silver", "gold"]:
    print(f"--- {ns} ---")
    spark.sql(f"SHOW TABLES IN {ns}").show(truncate=False)

In [ ]:
spark.sql("""
    SELECT kanal,
           COUNT(*)                        AS islem_adedi,
           ROUND(SUM(tutar) / 1e9, 2)      AS hacim_milyar_tl,
           COUNT(DISTINCT yatirimci_id)    AS tekil_yatirimci
    FROM silver.islem
    GROUP BY kanal
    ORDER BY hacim_milyar_tl DESC
""").show()

## Nessie: branch'ler ve commit geçmişi

Kendi deneyleriniz için **branch açın**. 0 byte kopyalar, main'i etkilemez:

```sql
CREATE BRANCH deneme_emir IN nessie FROM main;
USE REFERENCE deneme_emir IN nessie;
-- artık istediğinizi yazın, silin, bozun. main dokunulmaz.
```

> ### ⚠️ Önce bunu okuyun — branch'iniz ClickHouse'a sızabilir
>
> Yukarıdaki izolasyon **Spark için** geçerlidir. Ama bu kurulumda ClickHouse,
> Nessie katalogunu kullanamadığı için (`DataLakeCatalog` bu sürüm çiftinde
> veri okumuyor) `lake.*` görünümleri `icebergS3()` ile **doğrudan S3 yolunu**
> okuyor. `icebergS3()` katalogu atlar ve tablo dizinindeki **en yeni**
> `metadata.json`'ı seçer — bu **sizin branch'inizin** commit'i olabilir.
>
> Ölçüldü (`tests/06_clickhouse_branch_izolasyonu.py`): branch'e yazılan
> 1.000 satır ClickHouse'ta **göründü**; `DROP BRANCH` bile düzeltmedi
> (metadata S3'te kalıyor), ancak main'e yeni bir commit atılınca düzeldi.
>
> **Pratik kural:** Deneme branch'inizde **üretim tablolarına yazmayın.**
> Yeni bir tablo oluşturun (`silver.deneme_emir_x` gibi) — o zaman
> `lake.*` görünümleri etkilenmez, çünkü onlar başka dizinleri okuyor.
> Üretim tablosunu bozacaksanız önce kendi kopyanızı çıkarın:
>
> ```sql
> CREATE TABLE silver.deneme_emir AS SELECT * FROM silver.islem;
> ```
>
> Ayrıntı: `docs/01-mimari.md` bölüm 2.

In [ ]:
spark.sql("LIST REFERENCES IN nessie").show(truncate=False)

# 'SHOW LOG IN nessie' sutunlari:
#   author, committer, hash, message, signedOffBy, authorTime,
#   committerTime, properties
# DIKKAT: 'commitTime' diye bir sutun YOK. Dogrusu 'committerTime'.
log = spark.sql("SHOW LOG IN nessie")
log.select("hash", "message", "committerTime").show(5, truncate=False)

## Zamanda yolculuk

In [ ]:
# DIKKAT: '.snapshots' TUM snapshot'lari listeler -- rollback sonrasi artik
# gecerli olmayanlar dahil. Tablonun GUNCEL halini bulmak icin '.history'
# tablosunda is_current_ancestor = true olanlara bakin.
spark.sql(f"""
    SELECT snapshot_id, committed_at, operation,
           summary['added-records'] AS eklenen
    FROM {CATALOG}.silver.islem.snapshots
    ORDER BY committed_at DESC
""").show(truncate=False)

In [ ]:
# Belirli bir snapshot'taki hali okumak:
#   SELECT * FROM silver.islem VERSION AS OF <snapshot_id>
#   SELECT * FROM silver.islem TIMESTAMP AS OF '2026-03-31 18:00:00'
#
# Belirli bir Nessie branch'indeki hali okumak (backtick yeri onemli:
# namespace disarida, TABLO adi backtick icinde):
#   SELECT * FROM nessie.silver.`islem@branch_adi`
spark.sql(f"SELECT COUNT(*) AS guncel_satir FROM {CATALOG}.silver.`islem@main`").show()